# 14 — Planner & Orchestrator (Intent Router)

This notebook demonstrates the **IntentRouter** — NeuroForge's planner that:
1. Classifies natural language user inputs into intents
2. Extracts parameters (topic, difficulty, count, marks)
3. Routes to the appropriate workflow

Supports both **rule-based** (fast keyword matching) and **LLM-based** (complex inputs) classification.

## Supported Intents
| Intent | Triggers | Workflow |
|--------|----------|----------|
| quiz | quiz, test, exam, questions | QuizWorkflow |
| flashcard | flashcard, card, memory | FlashcardWorkflow |
| notes | notes, revision, summary | RevisionNotesWorkflow |
| explain | explain, what is, define | ChatTutor |
| solution | solution, solve, marks | SolutionWorkflow |
| mind_map | mind map, concept map | MindMapWorkflow |
| additional_info | application, interview, mistake | AdditionalInfoWorkflow |
| chat | (default) | ChatTutor |

In [ ]:
import sys
sys.path.insert(0, '..')

from src.planner import IntentRouter
from src.llm import LLMClient

print("IntentRouter loaded successfully!")

## 1. Initialize the Router

In [ ]:
# Initialize LLM client (for LLM-based classification)
llm_client = LLMClient()

# Create the intent router
router = IntentRouter(llm_client=llm_client)
print(f"Router initialized with {len(llm_client.available_providers)} LLM provider(s)")

## 2. Rule-Based Classification

Fast keyword matching — no LLM call needed. Handles clear, unambiguous inputs.

In [ ]:
# Test various natural language inputs with rule-based classification
test_inputs = [
    "Generate a quiz on photosynthesis",
    "Make 10 flashcards on machine learning",
    "Give me revision notes on calculus",
    "Explain what is osmosis",
    "Solve this 5 marks question on derivatives",
    "Create a mind map of data structures",
    "What are industry applications of blockchain?",
    "Hello, can you help me study?",
    "Generate 5 hard questions on quantum physics",
    "Give me interview questions on Python",
]

print("Rule-Based Intent Classification")
print("=" * 60)
for inp in test_inputs:
    result = router.classify_intent_rules(inp)
    print(f"\nInput: \"{inp}\"")
    print(f"  Intent: {result['intent']}")
    print(f"  Params: {result['parameters']}")

## 3. Parameter Extraction Examples

The router extracts topic, difficulty, count, and marks from natural language.

In [ ]:
param_examples = [
    "Generate 5 easy questions on biology",
    "Make 20 hard flashcards on organic chemistry",
    "Write a 10-mark answer on database normalization",
    "Create advanced notes on neural networks",
    "Give me simple quiz on addition",
]

print("Parameter Extraction")
print("=" * 60)
for inp in param_examples:
    result = router.classify_intent_rules(inp)
    params = result['parameters']
    print(f"\nInput: \"{inp}\"")
    print(f"  Topic:      {params.get('topic', '—')}")
    print(f"  Difficulty: {params.get('difficulty', '—')}")
    print(f"  Count:      {params.get('count', '—')}")
    print(f"  Marks:      {params.get('marks', '—')}")

## 4. LLM-Based Classification

For ambiguous inputs where keywords don't clearly indicate intent, the LLM provides better accuracy.

In [ ]:
# These inputs are ambiguous — rule-based would default to "chat"
ambiguous_inputs = [
    "I want to practice science",
    "Help me prepare for my biology exam tomorrow",
    "I need to memorize these chemistry formulas",
    "Can you test my knowledge of history?",
]

print("LLM-Based Intent Classification (ambiguous inputs)")
print("=" * 60)
for inp in ambiguous_inputs:
    try:
        result = router.classify_intent_llm(inp)
        print(f"\nInput: \"{inp}\"")
        print(f"  Intent: {result['intent']}")
        print(f"  Params: {result['parameters']}")
    except Exception as e:
        print(f"\nInput: \"{inp}\"")
        print(f"  Error: {e}")

## 5. Combined Classification (Rule-based + LLM Fallback)

The `classify_intent` method tries rules first, falls back to LLM for ambiguous cases.

In [ ]:
all_inputs = [
    # Clear intents (rule-based handles these)
    "Generate a quiz on photosynthesis",
    "Make flashcards for biology",
    "Summarize the chapter on thermodynamics",
    
    # Ambiguous (LLM fallback needed)
    "I want to practice science",
    "Help me memorize chemistry formulas",
    "What's the best way to learn calculus?",
]

print("Combined Classification (Rules + LLM Fallback)")
print("=" * 60)
for inp in all_inputs:
    result = router.classify_intent(inp)
    print(f"\nInput: \"{inp}\"")
    print(f"  Intent: {result['intent']}")
    print(f"  Params: {result['parameters']}")

## 6. Workflow Routing Demo

The router classifies intent and dispatches to the appropriate workflow.
Here we use mock workflows to demonstrate the routing logic.

In [ ]:
# Create mock workflows for demonstration
class MockWorkflow:
    """Simple mock workflow for routing demo."""
    def __init__(self, name):
        self.name = name
    
    def generate(self, topic, **kwargs):
        return f"[{self.name}] Generated content for '{topic}' with params: {kwargs}"

# Register workflows
workflows = {
    "quiz": MockWorkflow("QuizWorkflow"),
    "flashcard": MockWorkflow("FlashcardWorkflow"),
    "notes": MockWorkflow("RevisionNotesWorkflow"),
    "solution": MockWorkflow("SolutionWorkflow"),
    "mind_map": MockWorkflow("MindMapWorkflow"),
    "additional_info": MockWorkflow("AdditionalInfoWorkflow"),
    "chat": MockWorkflow("ChatTutor"),
}

# Route various inputs
routing_inputs = [
    "Generate 5 easy questions on photosynthesis",
    "Make flashcards on machine learning",
    "Give me revision notes on calculus",
    "Explain quantum entanglement",
    "Solve this 10 marks question on integration",
    "Create a mind map of sorting algorithms",
    "What are real world applications of AI?",
    "Hello, how are you?",
]

print("Workflow Routing Demo")
print("=" * 60)
for inp in routing_inputs:
    result = router.route(inp, workflows)
    print(f"\nInput: \"{inp}\"")
    print(f"  Result: {result}")

## 7. Integration with Real Workflows

Connecting the router to actual NeuroForge workflows.

In [ ]:
from src.workflows import (
    QuizWorkflow,
    RevisionNotesWorkflow,
    MindMapWorkflow,
    AdditionalInfoWorkflow,
    ChatTutor,
)

print("Real workflow imports available!")
print("In production, you would wire these up like:")
print()
print("  workflows = {")
print('      "quiz": QuizWorkflow(llm_client, retriever),')
print('      "notes": RevisionNotesWorkflow(llm_client, retriever),')
print('      "mind_map": MindMapWorkflow(llm_client, kg),')
print('      "additional_info": AdditionalInfoWorkflow(llm_client, retriever),')
print('      "chat": ChatTutor(llm_client, retriever),')
print("  }")
print()
print('  result = router.route("Generate 5 quiz questions on biology", workflows)')

## Summary

The **IntentRouter** provides:

- **Rule-based classification** — fast, no API calls, handles clear keywords
- **LLM-based classification** — handles ambiguous inputs with high accuracy
- **Combined strategy** — rules first, LLM fallback for unclear cases
- **Parameter extraction** — topic, difficulty, count, marks from natural language
- **Workflow routing** — dispatches to registered workflows with extracted params

### Architecture
```
User Input → IntentRouter
               ├── classify_intent_rules() — fast keyword match
               ├── classify_intent_llm() — LLM fallback
               └── route() → appropriate workflow.generate(topic, **params)
```